# Automação de Retreino com Alertas de Drift

Nesta aula, vamos fechar o ciclo de vida do nosso modelo de Machine Learning implementando uma **arquitetura de auto-correção**. Vamos adicionar uma nova etapa de alerta ao nosso *Job* principal (`carga_produtiva_monitoramento`) que, ao detectar um desvio de dados, acionará automaticamente o re-treinamento do modelo.

---

## Objetivos da Aula

* **Monitoramento Ativo:** Consultar os logs estruturados da tabela de auditoria para identificar degradação contínua.
* **Gatilho Condicional:** Desenvolver uma tarefa intermediária que toma decisões baseadas em condições.
* **Orquestração Dinâmica:** Acionar o pipeline de retreino injetando parâmetros de recorte temporal para garantir a consistência dos dados.


## Arquitetura do Workflow Atualizado

O nosso fluxo de engenharia de dados e MLOps seguirá a seguinte sequência:

1. **Carga Produtiva:** Ingestão das novas partições diárias de dados.
2. **Cálculo de Métricas:** Execução de testes estatísticos comparando a janela recente com a linha de base de treinamento.
3. **Log de Auditoria:** Gravação dos resultados na tabela `workspace.default.log_monitoramento_drift`.
4. **Validação e Alerta** Um script lê a tabela cruzando as datas da execução atual e avalia a coluna `drift_detectado`.
* **Se `false`:** A saúde do modelo está ok. O pipeline é encerrado normalmente.
* **Se `true`:** O alerta é acionado e a próxima tarefa é invocada.
5. **Retreino Automático:** O notebook `aula_5_5_re_treino` entra em execução. Ele consome a nova realidade dos dados, atualiza a matemática do modelo (via LightGBM/XGBoost) e promove a nova versão a `@Champion` no MLflow.

## Parâmetros e Variáveis de Ambiente

Para garantir que o modelo seja treinado exatamente sobre a janela anômala onde o problema ocorreu, o *Job* principal herda e repassa as seguintes variáveis de contexto (Widgets) para o notebook de retreino:

* **`data_inicio`**: Data inicial do recorte temporal (Formato: `YYYYMMDD`).
* **`data_fim`**: Data final do recorte temporal (Formato: `YYYYMMDD`).

---

## Scripts Relacionados

* **`carga_produtiva_monitoramento`**: *Job* orquestrador pai no Databricks Workflows.
* **`aula_5_5_re_treino`**: Notebook de modelagem responsável por ajustar os pesos do algoritmo e registrar o artefato atualizado no *Unity Catalog*.

## Prompt para o Genie

Aja como um Engenheiro de machine learning Sênior especialista em Databricks e PySpark.
Crie uma task para verificar se a tabela `workspace.default.log_monitoramento_drift` possui drift, para verificar se possui basta criar um notebook python que execute a seguinte query:
SELECT COUNT(*) as total_drift
FROM workspace.default.log_monitoramento_drift
WHERE drift_detectado = true
Se retormar valores, possui drift. Após isso crie uma task de condição para acionar outra task que é o re-treino, notebook chamado `aula_5_5_re_treino`.
O acionamento do notebook de re-treino deve ser feito passando os mesmos parâmetros `data_inicio` e `data_fim` para que ele treine o modelo no recorte temporal correto. Não alterar os jobs iniciais que já estão no fluxo.

